# Agent-4: Non-Markovian Dynamics and Quantum Combs

This notebook demonstrates how the Choi representation extends from a single quantum channel to a small multi-time process tensor, represented by a quantum comb Choi operator `T`.  We use the project convention

$$C_{\mathcal{E}} = \sum_{ij} |i\rangle\langle j| \otimes \mathcal{E}(|i\rangle\langle j|),$$

so the input system is the first tensor factor.  For the quantum comb `T`, subsystem order is `A0, B0, A1, B1, ...`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from combs_tools import (
    apply_choi_channel,
    blp_measure,
    choi_to_natural,
    comb_global_trace_preservation_check,
    deterministic_comb_causality_check,
    is_markovian,
    marginal_channel,
    natural_to_choi,
    rhp_measure,
    trace_distance,
)
from non_markovian_dynamics import (
    choi_abs_matrix,
    collision_model_comb,
    comb_correlation_norm,
    markovian_dephasing_family,
    memoryless_product_comb,
    pure_dephasing_choi,
)

np.random.seed(42)
PALETTE = {
    "blue": "#2F6BFF",
    "red": "#D1495B",
    "green": "#2A9D8F",
    "gold": "#F2B134",
    "ink": "#222222",
}
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Why a Single Channel Can Be Insufficient

For memoryless dynamics, the map from time 0 to time 2 factors through time 1 as a valid intermediate channel.  With environmental memory, each map from 0 to a fixed time can still be a perfectly valid CPTP channel, while the inferred intermediate map fails complete positivity.  This is the CP-divisibility viewpoint behind the RHP witness.

In [ ]:
def revival_dephasing_family(time: float) -> np.ndarray:
    """Positive coherence with a revival, avoiding singular zero crossings."""
    g = np.exp(-0.08 * time) * (0.58 + 0.42 * np.cos(3.0 * time))
    return pure_dephasing_choi(float(g))

t1, t2 = 0.65, 1.30
C_t1 = revival_dephasing_family(t1)
C_t2 = revival_dephasing_family(t2)

# A naive semigroup approximation would use E(t1) twice.
S_t1 = choi_to_natural(C_t1, 2, 2)
C_semigroup = natural_to_choi(S_t1 @ S_t1, 2, 2)
semigroup_error = np.linalg.norm(C_t2 - C_semigroup, ord="fro")
print(f"|| C(t2) - C(t1) composed with C(t1) ||_F = {semigroup_error:.4f}")

## 2. BLP and RHP-Style Grid Witnesses

The BLP estimate below sums positive revivals in trace distance after searching a finite grid of antipodal pure-state pairs, so it is a sampled approximation rather than the full state-pair optimization.  The RHP-style quantity is a discrete CP-divisibility witness: for each adjacent pair on `t_grid`, it reconstructs an intermediate map using a pseudo-inverse in natural representation and sums negative eigenvalues of the intermediate Choi matrix.  It should not be read as the continuous RHP integral.  The exponential dephasing family is CP-divisible, while the revival family is not.

In [ ]:
rho_plus = 0.5 * np.array([[1, 1], [1, 1]], dtype=complex)
rho_minus = 0.5 * np.array([[1, -1], [-1, 1]], dtype=complex)

t_grid = np.linspace(0.0, 3.0, 140)
markov_family = markovian_dephasing_family(rate=0.35)

def distance_curve(family):
    values = []
    for t in t_grid:
        C = family(float(t))
        values.append(trace_distance(apply_choi_channel(C, rho_plus), apply_choi_channel(C, rho_minus)))
    return np.array(values)

D_markov = distance_curve(markov_family)
D_revival = distance_curve(revival_dephasing_family)

blp_markov = blp_measure(markov_family, t_grid)
blp_revival = blp_measure(revival_dephasing_family, t_grid)
rhp_markov = rhp_measure(markov_family, t_grid)
rhp_revival = rhp_measure(revival_dephasing_family, t_grid)

print(f"BLP grid estimate, exponential dephasing: {blp_markov:.6f}")
print(f"BLP grid estimate, revival dephasing:     {blp_revival:.6f}")
print(f"RHP-style grid witness, exponential:      {rhp_markov:.6f}")
print(f"RHP-style grid witness, revival:          {rhp_revival:.6f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(t_grid, D_markov, color=PALETTE["blue"], label="CP-divisible exponential")
ax.plot(t_grid, D_revival, color=PALETTE["red"], label="revival family")
ax.set_xlabel("time")
ax.set_ylabel("trace distance D(rho_+(t), rho_-(t))")
ax.set_title("BLP intuition: information backflow appears as trace-distance revival")
ax.legend()
plt.show()

## 3. Quantum Comb as a Generalized Choi Operator

A two-use quantum comb `T` stores correlations between two time slots of a process tensor.  The helper `construct_process_tensor` builds the Choi operator of a finite-memory collision model: system slot 0 interacts with an environment, the same environment is carried forward, and then system slot 1 interacts with it.  The demonstration uses dense matrices and explicit basis loops, so it is intended for small qubit examples only.  If the comb were Markovian, it would factorize into the tensor product of its single-slot marginal Choi matrices.

In [ ]:
theta = 0.72
comb = collision_model_comb(theta=theta, n_steps=2)
product_comb = memoryless_product_comb(theta=theta, n_steps=2)

print(f"comb shape: {comb.shape}")
print(f"positive minimum eigenvalue: {np.linalg.eigvalsh(comb).min():.3e}")
print(f"global TP trace check: {comb_global_trace_preservation_check(comb, [2, 2, 2, 2])}")
print(f"deterministic comb causality hierarchy: {deterministic_comb_causality_check(comb, [2, 2, 2, 2])}")
print(f"factorizes into product marginals: {is_markovian(comb, n_steps=2, tol=1e-3)}")
print(f"relative comb correlation norm: {comb_correlation_norm(comb, n_steps=2):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), constrained_layout=True)
items = [
    (choi_abs_matrix(comb), "quantum comb |T|"),
    (choi_abs_matrix(product_comb), "product approximation"),
    (choi_abs_matrix(comb - product_comb), "correlation residue"),
]
for ax, (matrix, title) in zip(axes, items):
    im = ax.imshow(matrix, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 4. Marginal Channels and the Markovian Approximation

Taking a partial trace over all other time slots gives a single-slot marginal Choi matrix.  These marginals are useful, but they cannot represent temporal correlations by themselves.  The residue heatmap above is the part lost by replacing the comb with a product of marginal channels.

In [ ]:
C0 = marginal_channel(comb, 0)
C1 = marginal_channel(comb, 1)

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3), constrained_layout=True)
for ax, C, title in [(axes[0], C0, "slot 0 marginal |C0|"), (axes[1], C1, "slot 1 marginal |C1|")]:
    im = ax.imshow(choi_abs_matrix(C), cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

print("The marginals are valid single-time Choi matrices, but their product misses the memory residue.")

## 5. Takeaways

- A quantum comb `T` is a Choi-like operator for a multi-time process tensor.
- The BLP grid estimate detects memory as sampled trace-distance revival, often described as information backflow.
- The RHP-style grid witness detects memory as failure of CP-divisibility of pseudo-inverse reconstructed intermediate maps.
- The explicit comb representation makes the missing correlations visible when a process is replaced by a product of ordinary channels.
- The cost grows quickly: a qubit `N`-step comb already has matrix size `4**N` by `4**N`.